# Notebook 16: MechInt Pipeline & Database Infrastructure

**Duration**: 45-60 minutes  
**Prerequisites**: Notebooks 01-10 (basic familiarity), Python 3.8+, PyTorch

## Learning Objectives

By the end of this notebook, you will:

1. Understand the **MechIntDatabase** hybrid storage architecture (HDF5 + SQLite)
2. Store and query mechanistic interpretability results efficiently
3. Use **MechIntPipeline** to run comprehensive analysis workflows
4. Configure pipeline modes (quick, standard, comprehensive)
5. Implement custom analysis stages and generate reports
6. Leverage content-based deduplication and result caching

## Why Pipeline and Database Infrastructure?

Mechanistic interpretability research generates large amounts of heterogeneous data:

- **Circuits**: Graphs with nodes, edges, importance scores
- **Activation patterns**: High-dimensional tensors
- **Thermodynamic quantities**: Time series, energy flows, entropy production
- **Dynamics**: Trajectories, flow fields, phase space structures

Challenges:
- **Storage**: Efficient storage of mixed data types (arrays, metadata, graphs)
- **Retrieval**: Fast queries across experiments
- **Reproducibility**: Content-based hashing prevents duplicate computation
- **Workflow**: Coordinating multiple analysis stages

**Solution**: `MechIntDatabase` + `MechIntPipeline`

### Architecture

**MechIntDatabase**:
- HDF5 for large arrays (activations, trajectories)
- SQLite for metadata and fast queries
- Content-based hashing (SHA256) for deduplication
- Tagging system for experiment organization

**MechIntPipeline**:
- Configurable analysis stages
- Three preset modes: quick, standard, comprehensive
- Checkpointing and recovery
- Automatic report generation

## Key References

Database design:
- HDF5 documentation: https://www.hdfgroup.org/solutions/hdf5/
- SQLite documentation: https://www.sqlite.org/docs.html

Workflow systems:
- Prefect: https://www.prefect.io/
- Luigi: https://luigi.readthedocs.io/

Content-addressable storage:
- Git internals: https://git-scm.com/book/en/v2/Git-Internals-Plumbing-and-Porcelain

## Setup

In [ ]:
import sys
import tempfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import Dict, List, Optional, Any

# Ensure neuros_mechint is on path
# If running from examples directory, parent is neuros-mechint package
package_dir = Path.cwd().parent / 'src'
if package_dir.exists():
    sys.path.insert(0, str(package_dir))

from neuros_mechint.database import MechIntDatabase
from neuros_mechint.pipeline import MechIntPipeline, PipelineConfig
from neuros_mechint.circuits import AutomatedCircuitDiscovery
from neuros_mechint.energy_flow import LandauerAnalyzer
from neuros_mechint.dynamics import NeuralODEIntegrator
from neuros_mechint.visualization import EnhancedVisualizer

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Matplotlib style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

## Part 1: MechIntDatabase Basics

The database uses a hybrid architecture:
- **SQLite**: Fast queries on metadata (tags, timestamps, result types)
- **HDF5**: Efficient storage of large arrays
- **Content hashing**: SHA256 hash of inputs prevents duplicate computation

### 1.1 Database Setup and Storage

In [ ]:
# Create temporary database for demonstration
temp_dir = tempfile.mkdtemp()
db = MechIntDatabase(root_dir=temp_dir)

print(f"Database created at: {temp_dir}")
print(f"SQLite database: {Path(temp_dir) / 'mechint.db'}")
print(f"HDF5 storage: {Path(temp_dir) / 'results.h5'}")

### 1.2 Storing Analysis Results

Let's run some analyses and store the results.

In [ ]:
# Create a simple model for demonstration
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim: int = 20, hidden_dim: int = 32, output_dim: int = 3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

model = SimpleClassifier().to(device)
model.eval()

# Generate sample data
batch_size = 32
input_dim = 20
sample_inputs = torch.randn(batch_size, input_dim, device=device)
sample_labels = torch.randint(0, 3, (batch_size,), device=device)

print(f"Model: {model.__class__.__name__}")
print(f"Sample inputs: {sample_inputs.shape}")

In [ ]:
# Run ACDC analysis
print("Running ACDC analysis...")
acdc = AutomatedCircuitDiscovery(
    model=model,
    importance_threshold=0.05,
    device=device
)

circuit_result = acdc.discover_circuit(
    inputs=sample_inputs[:16],  # Use subset for speed
    targets=sample_labels[:16],
    max_iterations=10
)

# Store in database with tags
circuit_id = db.store(
    result=circuit_result,
    tags=['acdc', 'simple_classifier', 'experiment_1']
)

print(f"✓ Circuit stored with ID: {circuit_id}")
print(f"  - Nodes: {len(circuit_result.nodes)}")
print(f"  - Edges: {len(circuit_result.edges)}")
print(f"  - Tags: {db.get_tags(circuit_id)}")

In [ ]:
# Run Landauer analysis
print("\nRunning Landauer analysis...")
landauer = LandauerAnalyzer(
    model=model,
    temperature=300.0,
    device=device
)

landauer_result = landauer.analyze_forward_pass(inputs=sample_inputs)

# Store with different tags
landauer_id = db.store(
    result=landauer_result,
    tags=['landauer', 'simple_classifier', 'experiment_1', 'thermodynamics']
)

print(f"✓ Landauer analysis stored with ID: {landauer_id}")
print(f"  - Bits erased: {landauer_result.total_bits_erased:.2f}")
print(f"  - Min energy: {landauer_result.total_min_energy:.2e} J")
print(f"  - Tags: {db.get_tags(landauer_id)}")

### 1.3 Content-Based Deduplication

The database uses content hashing to detect duplicate computations.

In [ ]:
# Try to store the same circuit result again
print("Attempting to store duplicate result...")
duplicate_id = db.store(
    result=circuit_result,
    tags=['acdc', 'duplicate_test']
)

print(f"Original ID:  {circuit_id}")
print(f"Duplicate ID: {duplicate_id}")
print(f"Same? {circuit_id == duplicate_id}")

# Tags are merged for duplicate content
print(f"\nMerged tags: {db.get_tags(circuit_id)}")

### 1.4 Querying Results

The database supports tag-based queries and retrieval.

In [ ]:
# Query by tag
print("Querying all 'experiment_1' results:")
exp1_ids = db.query(tags=['experiment_1'])
print(f"Found {len(exp1_ids)} results")

for result_id in exp1_ids:
    result = db.load(result_id)
    print(f"  - {result_id}: {type(result).__name__}")

# Query thermodynamic results
print("\nQuerying 'thermodynamics' results:")
thermo_ids = db.query(tags=['thermodynamics'])
print(f"Found {len(thermo_ids)} results")

# Query with multiple tags (AND logic)
print("\nQuerying 'simple_classifier' AND 'acdc' results:")
classifier_acdc_ids = db.query(tags=['simple_classifier', 'acdc'])
print(f"Found {len(classifier_acdc_ids)} results")

### 1.5 Database Statistics

In [ ]:
# Get database statistics
all_ids = db.list_all()
print(f"Total stored results: {len(all_ids)}")

# Group by result type
from collections import defaultdict
type_counts = defaultdict(int)

for result_id in all_ids:
    result = db.load(result_id)
    type_counts[type(result).__name__] += 1

print("\nResults by type:")
for result_type, count in sorted(type_counts.items()):
    print(f"  {result_type}: {count}")

# Get all unique tags
all_tags = set()
for result_id in all_ids:
    all_tags.update(db.get_tags(result_id))

print(f"\nUnique tags: {sorted(all_tags)}")

## Part 2: MechIntPipeline Configuration

The pipeline orchestrates multi-stage analysis workflows.

### Pipeline Modes

1. **Quick** (~5 min): Basic circuit discovery + energy analysis
2. **Standard** (~15 min): + Thermodynamics + Dynamics
3. **Comprehensive** (~30+ min): + Advanced comparisons + Full visualization

### 2.1 Quick Pipeline Mode

In [ ]:
# Configure quick pipeline
quick_config = PipelineConfig(
    mode='quick',
    enable_circuits=True,
    enable_energy=True,
    enable_thermodynamics=False,
    enable_dynamics=False,
    enable_visualization=True,
    save_checkpoints=True
)

# Create pipeline
quick_pipeline = MechIntPipeline(
    config=quick_config,
    database=db,
    device=device
)

print("Quick Pipeline Configuration:")
print(f"  Mode: {quick_config.mode}")
print(f"  Circuits: {quick_config.enable_circuits}")
print(f"  Energy: {quick_config.enable_energy}")
print(f"  Thermodynamics: {quick_config.enable_thermodynamics}")
print(f"  Dynamics: {quick_config.enable_dynamics}")
print(f"  Visualization: {quick_config.enable_visualization}")

In [ ]:
# Run quick pipeline
print("Running quick pipeline...")
quick_results = quick_pipeline.run(
    model=model,
    inputs=sample_inputs,
    targets=sample_labels,
    experiment_name='quick_demo'
)

print(f"\n✓ Quick pipeline complete!")
print(f"Results: {list(quick_results.keys())}")
print(f"\nStored result IDs:")
for stage, result_id in quick_results.items():
    print(f"  {stage}: {result_id}")

### 2.2 Standard Pipeline Mode

In [ ]:
# Configure standard pipeline (more comprehensive)
standard_config = PipelineConfig(
    mode='standard',
    enable_circuits=True,
    enable_energy=True,
    enable_thermodynamics=True,
    enable_dynamics=True,
    enable_visualization=True,
    save_checkpoints=True
)

standard_pipeline = MechIntPipeline(
    config=standard_config,
    database=db,
    device=device
)

print("Standard Pipeline Configuration:")
print(f"  Mode: {standard_config.mode}")
print(f"  All stages enabled: {all([standard_config.enable_circuits, standard_config.enable_thermodynamics, standard_config.enable_dynamics])}")

In [ ]:
# Run standard pipeline
print("Running standard pipeline...")
standard_results = standard_pipeline.run(
    model=model,
    inputs=sample_inputs,
    targets=sample_labels,
    experiment_name='standard_demo'
)

print(f"\n✓ Standard pipeline complete!")
print(f"Results: {list(standard_results.keys())}")
print(f"\nAdditional stages vs quick:")
new_stages = set(standard_results.keys()) - set(quick_results.keys())
print(f"  {new_stages}")

### 2.3 Custom Pipeline Configuration

You can create custom pipelines with specific analysis stages.

In [ ]:
# Custom configuration: Only thermodynamics + visualization
custom_config = PipelineConfig(
    mode='custom',
    enable_circuits=False,
    enable_energy=False,
    enable_thermodynamics=True,
    enable_dynamics=False,
    enable_visualization=True,
    save_checkpoints=False  # Disable checkpointing for speed
)

custom_pipeline = MechIntPipeline(
    config=custom_config,
    database=db,
    device=device
)

print("Custom Pipeline - Thermodynamics Only:")
custom_results = custom_pipeline.run(
    model=model,
    inputs=sample_inputs,
    targets=sample_labels,
    experiment_name='custom_thermo'
)

print(f"Results: {list(custom_results.keys())}")

## Part 3: End-to-End Workflow

Let's demonstrate a complete research workflow:
1. Train multiple model variants
2. Run comprehensive analysis pipeline
3. Query and compare results
4. Generate comparative visualizations

### 3.1 Create Model Variants

In [ ]:
# Create three model variants: Wide, Deep, Balanced
models = {
    'wide': SimpleClassifier(input_dim=20, hidden_dim=64, output_dim=3).to(device),
    'deep': nn.Sequential(
        nn.Linear(20, 24),
        nn.ReLU(),
        nn.Linear(24, 24),
        nn.ReLU(),
        nn.Linear(24, 24),
        nn.ReLU(),
        nn.Linear(24, 24),
        nn.ReLU(),
        nn.Linear(24, 3)
    ).to(device),
    'balanced': SimpleClassifier(input_dim=20, hidden_dim=32, output_dim=3).to(device)
}

for name, model in models.items():
    model.eval()
    param_count = sum(p.numel() for p in model.parameters())
    print(f"{name:10s}: {param_count:,} parameters")

### 3.2 Run Comprehensive Analysis on All Models

In [ ]:
# Use comprehensive pipeline configuration
comprehensive_config = PipelineConfig(
    mode='comprehensive',
    enable_circuits=True,
    enable_energy=True,
    enable_thermodynamics=True,
    enable_dynamics=True,
    enable_visualization=True,
    save_checkpoints=True
)

comprehensive_pipeline = MechIntPipeline(
    config=comprehensive_config,
    database=db,
    device=device
)

# Store results for each model
model_results = {}

for model_name, model in models.items():
    print(f"\nAnalyzing {model_name} model...")
    results = comprehensive_pipeline.run(
        model=model,
        inputs=sample_inputs,
        targets=sample_labels,
        experiment_name=f'comparison_{model_name}'
    )
    model_results[model_name] = results
    print(f"  ✓ {len(results)} stages completed")

print("\n" + "="*50)
print("All models analyzed!")
print(f"Total results in database: {len(db.list_all())}")

### 3.3 Query and Compare Results

In [ ]:
# Query all Landauer results
landauer_ids = db.query(tags=['landauer'])
print(f"Found {len(landauer_ids)} Landauer analyses\n")

# Compare energy costs across models
energy_comparison = {}

for result_id in landauer_ids:
    result = db.load(result_id)
    tags = db.get_tags(result_id)
    
    # Extract model name from tags
    model_tag = [t for t in tags if t.startswith('comparison_')]
    if model_tag:
        model_name = model_tag[0].replace('comparison_', '')
        energy_comparison[model_name] = {
            'bits_erased': result.total_bits_erased,
            'min_energy': result.total_min_energy,
            'reversibility': result.reversibility_score
        }

print("Energy Cost Comparison:")
print(f"{'Model':<12} {'Bits Erased':>12} {'Min Energy (J)':>15} {'Reversibility':>15}")
print("-" * 60)
for model_name, metrics in sorted(energy_comparison.items()):
    print(f"{model_name:<12} {metrics['bits_erased']:>12.2f} {metrics['min_energy']:>15.2e} {metrics['reversibility']:>15.3f}")

### 3.4 Visualize Comparative Results

In [ ]:
# Extract circuit complexity across models
circuit_ids = db.query(tags=['acdc'])
circuit_comparison = {}

for result_id in circuit_ids:
    result = db.load(result_id)
    tags = db.get_tags(result_id)
    
    model_tag = [t for t in tags if t.startswith('comparison_')]
    if model_tag:
        model_name = model_tag[0].replace('comparison_', '')
        circuit_comparison[model_name] = {
            'nodes': len(result.nodes),
            'edges': len(result.edges),
            'iterations': result.n_iterations
        }

# Create comparative visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Model Comparison: Circuits vs Thermodynamics', fontsize=14, fontweight='bold')

# Plot 1: Circuit Complexity
ax = axes[0, 0]
models_list = list(circuit_comparison.keys())
nodes = [circuit_comparison[m]['nodes'] for m in models_list]
edges = [circuit_comparison[m]['edges'] for m in models_list]

x = np.arange(len(models_list))
width = 0.35
ax.bar(x - width/2, nodes, width, label='Nodes', alpha=0.8)
ax.bar(x + width/2, edges, width, label='Edges', alpha=0.8)
ax.set_xlabel('Model')
ax.set_ylabel('Count')
ax.set_title('Circuit Complexity')
ax.set_xticks(x)
ax.set_xticklabels(models_list)
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Energy Cost
ax = axes[0, 1]
models_list = list(energy_comparison.keys())
energy_vals = [energy_comparison[m]['min_energy'] * 1e21 for m in models_list]  # Convert to 10^-21 J
ax.bar(models_list, energy_vals, alpha=0.8, color='coral')
ax.set_xlabel('Model')
ax.set_ylabel('Minimum Energy (×10⁻²¹ J)')
ax.set_title('Thermodynamic Cost')
ax.grid(True, alpha=0.3)

# Plot 3: Reversibility
ax = axes[1, 0]
reversibility = [energy_comparison[m]['reversibility'] for m in models_list]
colors = ['green' if r > 0.5 else 'orange' if r > 0.3 else 'red' for r in reversibility]
ax.bar(models_list, reversibility, alpha=0.8, color=colors)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Target')
ax.set_xlabel('Model')
ax.set_ylabel('Reversibility Score')
ax.set_title('Computational Reversibility')
ax.set_ylim([0, 1])
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Efficiency Ratio (Nodes per Energy)
ax = axes[1, 1]
efficiency = []
for m in models_list:
    # Circuit efficiency: more nodes with less energy is better
    eff = circuit_comparison[m]['nodes'] / (energy_comparison[m]['min_energy'] * 1e21)
    efficiency.append(eff)

ax.bar(models_list, efficiency, alpha=0.8, color='steelblue')
ax.set_xlabel('Model')
ax.set_ylabel('Nodes per (10⁻²¹ J)')
ax.set_title('Circuit Efficiency')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInsights:")
best_efficiency_idx = np.argmax(efficiency)
print(f"  - Most efficient architecture: {models_list[best_efficiency_idx]}")
print(f"  - Most reversible: {models_list[np.argmax(reversibility)]}")
print(f"  - Simplest circuit: {models_list[np.argmin(edges)]}")

## Part 4: Advanced Features

### 4.1 Checkpoint and Recovery

Long-running pipelines can be interrupted and resumed.

In [ ]:
# Demonstrate checkpoint functionality
checkpoint_config = PipelineConfig(
    mode='standard',
    enable_circuits=True,
    enable_energy=True,
    enable_thermodynamics=True,
    enable_dynamics=True,
    enable_visualization=True,
    save_checkpoints=True
)

checkpoint_pipeline = MechIntPipeline(
    config=checkpoint_config,
    database=db,
    device=device
)

# Check if checkpoint exists
experiment_name = 'checkpoint_demo'
checkpoint_exists = checkpoint_pipeline.has_checkpoint(experiment_name)

print(f"Checkpoint exists for '{experiment_name}': {checkpoint_exists}")

if checkpoint_exists:
    print("Resuming from checkpoint...")
    results = checkpoint_pipeline.resume_from_checkpoint(experiment_name)
else:
    print("Running new pipeline...")
    results = checkpoint_pipeline.run(
        model=models['balanced'],
        inputs=sample_inputs,
        targets=sample_labels,
        experiment_name=experiment_name
    )

print(f"\nCompleted stages: {list(results.keys())}")

### 4.2 Pipeline Report Generation

Generate a comprehensive report of pipeline results.

In [ ]:
# Generate report for one of our experiments
report = checkpoint_pipeline.generate_report(
    results=model_results['balanced'],
    experiment_name='balanced_model_analysis'
)

print("Pipeline Report:")
print("=" * 60)
print(report)
print("=" * 60)

### 4.3 Custom Analysis Stages

You can add custom analysis stages to the pipeline.

In [ ]:
# Define a custom analysis function
@dataclass
class CustomMetricResult:
    """Custom analysis result."""
    metric_name: str
    value: float
    metadata: Dict[str, Any]

def custom_sparsity_analysis(
    model: nn.Module,
    inputs: torch.Tensor,
    device: torch.device
) -> CustomMetricResult:
    """Analyze activation sparsity in the model."""
    activations = []
    
    def hook(module, input, output):
        if isinstance(output, torch.Tensor):
            activations.append(output.detach())
    
    # Register hooks
    hooks = []
    for module in model.modules():
        if isinstance(module, nn.ReLU):
            hooks.append(module.register_forward_hook(hook))
    
    # Forward pass
    with torch.no_grad():
        model(inputs)
    
    # Remove hooks
    for h in hooks:
        h.remove()
    
    # Calculate sparsity (fraction of zeros)
    if activations:
        all_activations = torch.cat([a.flatten() for a in activations])
        sparsity = (all_activations == 0).float().mean().item()
    else:
        sparsity = 0.0
    
    return CustomMetricResult(
        metric_name='activation_sparsity',
        value=sparsity,
        metadata={
            'n_relu_layers': len(activations),
            'total_activations': sum(a.numel() for a in activations)
        }
    )

# Run custom analysis
print("Running custom sparsity analysis...")
sparsity_result = custom_sparsity_analysis(
    model=models['balanced'],
    inputs=sample_inputs,
    device=device
)

# Store in database
sparsity_id = db.store(
    result=sparsity_result,
    tags=['custom_analysis', 'sparsity', 'balanced']
)

print(f"\n✓ Custom analysis complete")
print(f"  Metric: {sparsity_result.metric_name}")
print(f"  Value: {sparsity_result.value:.3f}")
print(f"  ReLU layers analyzed: {sparsity_result.metadata['n_relu_layers']}")
print(f"  Stored with ID: {sparsity_id}")

## Summary

In this notebook, we explored the **MechIntDatabase** and **MechIntPipeline** infrastructure:

### Key Concepts

1. **Hybrid Storage**:
   - SQLite: Fast metadata queries
   - HDF5: Efficient array storage
   - Content hashing: Automatic deduplication

2. **Tag-Based Organization**:
   ```python
   db.store(result, tags=['experiment', 'acdc', 'model_v1'])
   results = db.query(tags=['experiment', 'acdc'])  # AND logic
   ```

3. **Pipeline Modes**:
   - Quick: Fast iteration (~5 min)
   - Standard: Balanced analysis (~15 min)
   - Comprehensive: Full analysis (~30+ min)
   - Custom: User-defined stages

4. **Workflow Benefits**:
   - Reproducibility: Content-based deduplication
   - Efficiency: Checkpoint and resume
   - Organization: Tag-based retrieval
   - Extensibility: Custom analysis stages

### When to Use Each Component

**Use MechIntDatabase when**:
- Running multiple experiments
- Need to compare results across models
- Want automatic deduplication
- Require fast queries on metadata

**Use MechIntPipeline when**:
- Running standardized analysis workflows
- Need checkpoint/resume capability
- Want automated report generation
- Coordinating multiple analysis stages

### Best Practices

1. **Tag Strategically**: Use hierarchical tags (e.g., `experiment_name`, `model_type`, `analysis_type`)
2. **Enable Checkpoints**: For long-running pipelines
3. **Query Before Compute**: Check if result already exists
4. **Custom Stages**: Extend pipeline with domain-specific analyses
5. **Regular Cleanup**: Archive old experiments to manage storage

## Exercises

### Exercise 1: Custom Pipeline for Sparse Models

Create a custom pipeline configuration that focuses on sparsity analysis:
1. Disable thermodynamics and dynamics
2. Enable circuits and energy
3. Add custom sparsity analysis stage
4. Run on a model with L1 regularization
5. Compare sparsity vs energy cost

**Starter code:**

In [ ]:
# Implement sparse model pipeline
def create_sparse_pipeline(db, device):
    # TODO: Create custom config for sparsity-focused analysis
    config = PipelineConfig(
        mode='custom',
        # Add appropriate flags
    )
    # TODO: Return configured pipeline
    pass

# TODO: Create sparse model with L1 regularization
# TODO: Run pipeline
# TODO: Compare sparsity vs energy efficiency

### Exercise 2: Database Query Optimization

Implement efficient queries for a large-scale experiment:
1. Store results from 20+ model variants
2. Query all results from a specific date range
3. Find top-5 most efficient models (circuit nodes / energy)
4. Generate comparative report

**Starter code:**

In [ ]:
# Implement efficient querying
def find_top_efficient_models(db, n_top=5):
    # TODO: Query all circuit and energy results
    # TODO: Calculate efficiency metric
    # TODO: Return top N models with metadata
    pass

# TODO: Implement date range filtering
# TODO: Generate comparative visualization

### Exercise 3: Batch Analysis Pipeline

Create a pipeline that analyzes multiple models in parallel:
1. Define 5 model architectures (varying depth/width)
2. Run comprehensive pipeline on each
3. Store all results with consistent tagging
4. Query and visualize trade-offs:
   - Model complexity vs circuit complexity
   - Parameter count vs energy cost
   - Reversibility vs accuracy

**Starter code:**

In [ ]:
# Implement batch analysis
def batch_analyze_models(models_dict, pipeline, db, inputs, targets):
    # TODO: Run pipeline on each model
    # TODO: Store with consistent tags (include timestamp, architecture)
    # TODO: Return aggregated results
    pass

# TODO: Create 5 model variants
# TODO: Run batch analysis
# TODO: Query results
# TODO: Create multi-dimensional comparison plots

## Next Steps

You've completed the Phase 2 example notebooks! Here's what to explore next:

1. **Integration**: Combine techniques from multiple notebooks
2. **Real Models**: Apply to production architectures (ResNet, Transformers, etc.)
3. **Custom Analyzers**: Implement domain-specific interpretability methods
4. **Visualization**: Create interactive Bokeh dashboards for exploration
5. **Research**: Use infrastructure for novel mechanistic interpretability research

Recommended reading order:
- Notebooks 01-05: Core foundations
- Notebooks 06-10: Advanced SAE analysis  
- Notebooks 11-13: Circuit discovery
- Notebooks 14-15: Dynamics and energy
- **Notebook 16** (this one): Infrastructure

For comprehensive Phase 2 guidance, see `PHASE2_GUIDE.md` in the examples directory.